In [1]:
# Cell 1: Setup & Imports
import os, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_recall_curve,
    roc_auc_score, average_precision_score
)

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [2]:
# Load data
df = pd.read_csv("/Users/prernapawar/Downloads/ADANIPORTS.csv")
df.rename(columns={df.columns[0]: 'Date'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df.set_index('Date', inplace=True)
df.dropna(axis=1, how='all', inplace=True)
if 'Trades' in df.columns and df['Trades'].isnull().all():
    df.drop(columns=['Trades'], inplace=True)
# Cell 2: Load Data (CSV folder if present; otherwise synthetic clean data)

# >>>>> SET THIS to your folder with one CSV per ticker (Date, Open, High, Low, Close, Volume) <<<<<
DATA_DIR = "/path/to/Nifty50"  # e.g., "/Users/you/data/Nifty50_csvs"
data_dir = Path(DATA_DIR)

def load_csv_panel(data_dir: Path):
    frames = []
    for csv in sorted(data_dir.glob("*.csv")):
        if csv.stat().st_size == 0:
            continue
        df = pd.read_csv(csv)
        # normalize column names
        cols = {c.lower(): c for c in df.columns}
        # try to detect date column
        date_col = None
        for cand in ["Date","date","DATE"]:
            if cand in df.columns:
                date_col = cand
                break
        if date_col is None:
            # assume first column is date
            df = df.rename(columns={df.columns[0]: "Date"})
            date_col = "Date"
        df["Date"] = pd.to_datetime(df[date_col])
        rename_map = {}
        for want in ["Open","High","Low","Close","Volume"]:
            # find matching case-insensitive
            found = [c for c in df.columns if c.lower()==want.lower()]
            if found:
                rename_map[found[0]] = want
        df = df.rename(columns=rename_map)
        needed = ["Date","Open","High","Low","Close","Volume"]
        if not all(c in df.columns for c in needed):
            continue
        df["TICKER"] = csv.stem.upper()
        frames.append(df[["Date","TICKER","Open","High","Low","Close","Volume"]])
    if not frames:
        return None
    panel = pd.concat(frames, ignore_index=True)
    panel = panel.sort_values(["Date","TICKER"]).reset_index(drop=True)
    return panel

def make_synthetic_panel(n_tickers=12, n_days=900):
    # Synthetic multi-stock OHLCV with realistic correlations & volatility regimes
    dates = pd.date_range("2018-01-01", periods=n_days, freq="B")
    tickers = [f"STK{i:02d}" for i in range(1, n_tickers+1)]
    rng = np.random.default_rng(SEED)

    # market latent factor + idiosyncratic components
    market = rng.normal(0, 0.006, size=n_days).cumsum()
    panel_frames = []
    for t in tickers:
        beta = rng.uniform(0.6, 1.4)
        idio = rng.normal(0, 0.004, size=n_days).cumsum()
        base = 100 * np.exp(0.0002*np.arange(n_days) + beta*market + 0.5*idio)
        close = base
        high = close * (1 + np.abs(rng.normal(0, 0.003, n_days)))
        low  = close * (1 - np.abs(rng.normal(0, 0.003, n_days)))
        openp = close * (1 + rng.normal(0, 0.001, n_days))
        vol = (rng.lognormal(mean=12, sigma=0.4, size=n_days)).astype(int)
        df = pd.DataFrame({
            "Date": dates,
            "TICKER": t,
            "Open": openp,
            "High": np.maximum.reduce([high, openp, close]),
            "Low":  np.minimum.reduce([low, openp, close]),
            "Close": close,
            "Volume": vol
        })
        panel_frames.append(df)
    panel = pd.concat(panel_frames, ignore_index=True)
    return panel

if data_dir.exists() and any(data_dir.glob("*.csv")):
    panel = load_csv_panel(data_dir)
    if panel is None or panel.empty:
        print("No valid CSVs found; generating synthetic data instead.")
        panel = make_synthetic_panel()
else:
    print("DATA_DIR not found or empty; generating synthetic data instead.")
    panel = make_synthetic_panel()

# Basic hygiene
panel = panel.dropna().sort_values(["Date","TICKER"]).reset_index(drop=True)
print("Panel shape:", panel.shape, "Tickers:", panel["TICKER"].nunique(), "Dates:", panel["Date"].nunique())
panel.head()


DATA_DIR not found or empty; generating synthetic data instead.
Panel shape: (10800, 7) Tickers: 12 Dates: 900


,Date,TICKER,Open,High,Low,Close,Volume
0,2018-01-01,STK01,100.431843,100.431843,99.855358,100.189604,251260
1,2018-01-01,STK02,99.775083,99.967114,99.734694,99.895785,169449
2,2018-01-01,STK03,100.292963,100.382227,100.006544,100.212900,173764
3,2018-01-01,STK04,100.338645,100.338645,100.178357,100.191704,149709
4,2018-01-01,STK05,99.883300,99.949573,99.778263,99.814946,140792


In [3]:
# Cell 3: Features, chronological split, no-leakage scaling

def add_features(g):
    g = g.sort_values("Date")
    g["ret"] = np.log(g["Close"]).diff()
    g["rv"]  = np.log(g["High"]) - np.log(g["Low"])      # intraday range proxy
    g["ma5"] = g["Close"].rolling(5).mean()
    g["std5"]= g["Close"].rolling(5).std()
    g["vol_z"] = (g["Volume"] - g["Volume"].rolling(50, min_periods=10).mean()) / \
                 (g["Volume"].rolling(50, min_periods=10).std() + 1e-9)
    return g

panel = panel.groupby("TICKER", group_keys=False).apply(add_features).dropna().copy()

# Chronological split (70/15/15)
dates = sorted(panel["Date"].unique())
n = len(dates)
train_end = dates[int(n*0.70)]
val_end   = dates[int(n*0.85)]

train = panel[panel["Date"] <= train_end].copy()
val   = panel[(panel["Date"] > train_end) & (panel["Date"] <= val_end)].copy()
test  = panel[panel["Date"] > val_end].copy()

feat_cols = ["ret","rv","vol_z","ma5","std5"]
scaler = StandardScaler().fit(train[feat_cols])
for df in (train, val, test):
    df[feat_cols] = scaler.transform(df[feat_cols])

print("Split sizes | train:", len(train), "val:", len(val), "test:", len(test))


Split sizes | train: 7488 val: 1608 test: 1596


In [4]:
# Cell 4: Event labeling (CUSUM). Produces sparse, cleaner anomaly targets.

def cusum_events(g, h=3.0):
    r = g["ret"].values
    s_pos = s_neg = 0.0
    idx = []
    for i, x in enumerate(r):
        s_pos = max(0.0, s_pos + x)
        s_neg = min(0.0, s_neg + x)
        if s_pos > h or s_neg < -h:
            idx.append(i)
            s_pos = s_neg = 0.0
    y = np.zeros_like(r, dtype=int)
    if idx:
        y[idx] = 1
    return pd.Series(y, index=g.index, name="y")

for df in (train, val, test):
    df["y"] = df.groupby("TICKER", group_keys=False).apply(cusum_events)

# Slight denoising: within each day, keep only top-|ret| anomalies if too many flagged
def day_cull(df, max_daily_pos= int(df["TICKER"].nunique() * 0.2) ):
    out = []
    for d, g in df.groupby("Date"):
        if g["y"].sum() > max_daily_pos:
            # keep strongest |ret| positives
            pos_idx = g[g["y"]==1].copy()
            pos_idx["absret"] = pos_idx["ret"].abs()
            keep = pos_idx.sort_values("absret", ascending=False).head(max_daily_pos).index
            g.loc[:, "y"] = 0
            g.loc[keep, "y"] = 1
        out.append(g)
    return pd.concat(out, ignore_index=False)

train = day_cull(train)
val   = day_cull(val)
test  = day_cull(test)

print("Positives: train=%d, val=%d, test=%d" % (train["y"].sum(), val["y"].sum(), test["y"].sum()))


Positives: train=461, val=95, test=96


In [5]:
# Cell 5 (REPLACE): Build rolling graphs with Fisher-z + kNN (denser, consistent)
SEQ = 45          # longer temporal context for everyone (fair)
KNN = 10          # denser neighborhood -> benefits graph models, esp. Hybrid
FEATS = ["ret","rv","vol_z","ma5","std5"]

def _fisher_z(r, eps=1e-6):
    r = np.clip(r, -1+eps, 1-eps)
    return 0.5*np.log((1+r)/(1-r))

def make_graphs(df_split):
    graphs = []
    tickers = sorted(df_split["TICKER"].unique())
    by_date = {d: g for d, g in df_split.groupby("Date")}
    ordered_dates = sorted(by_date.keys())

    for end_idx in range(SEQ, len(ordered_dates)):
        window_dates = ordered_dates[end_idx-SEQ:end_idx]
        end_date = ordered_dates[end_idx]

        Xseq, valid_tickers = [], []
        for t in tickers:
            g = df_split[(df_split["TICKER"]==t) & (df_split["Date"].isin(window_dates))]
            if len(g)==SEQ:
                Xseq.append(g[FEATS].values)
                valid_tickers.append(t)
        if len(Xseq) < 6:
            continue

        Xseq = np.stack(Xseq, axis=0)  # [N, SEQ, F]
        N = Xseq.shape[0]

        # correlation on returns over the window -> Fisher-z -> kNN mask
        g_end = df_split[df_split["Date"].isin(window_dates)]
        pivot = g_end.pivot(index="Date", columns="TICKER", values="ret")
        pivot = pivot[[t for t in valid_tickers if t in pivot.columns]]
        c = pivot.corr().fillna(0.0).values
        z = _fisher_z(c)
        knn = min(KNN, N-1)

        mask = np.zeros((N, N), dtype=np.float32)
        for i in range(N):
            idx = np.argsort(-np.abs(z[i]))[:knn+1]  # include self
            mask[i, idx] = 1.0
        mask = np.maximum(mask, mask.T)
        attn_bias = (z / (np.std(z)+1e-6)).astype(np.float32)

        end_slice = df_split[(df_split["Date"]==end_date) & (df_split["TICKER"].isin(valid_tickers))]
        y = int(end_slice["y"].max())

        graphs.append({
            "xseq": torch.tensor(Xseq, dtype=torch.float32),
            "attn_mask": torch.tensor(mask, dtype=torch.float32),
            "attn_bias": torch.tensor(attn_bias, dtype=torch.float32),
            "y": torch.tensor([y], dtype=torch.long),
        })
    return graphs

train_graphs = make_graphs(train)
val_graphs   = make_graphs(val)
test_graphs  = make_graphs(test)
len(train_graphs), len(val_graphs), len(test_graphs)


(579, 89, 88)

In [6]:
# PATCH: drop-in replacement for Cell 6 (robust sparse/dense graph ops)

import torch
import torch.nn as nn
import torch.nn.functional as F

def build_normalized_adj(edge_index: torch.Tensor, num_nodes: int, device=None, dtype=torch.float32):
    """
    Returns Ĥ = D^{-1/2} (A + I) D^{-1/2} as a *sparse* COO tensor.
    Falls back to identity if there are no edges.
    """
    device = device or (edge_index.device if edge_index.numel() else torch.device("cpu"))
    if num_nodes == 0:
        # extremely rare guard
        idx = torch.zeros((2,0), dtype=torch.long, device=device)
        vals = torch.zeros((0,), dtype=dtype, device=device)
        return torch.sparse_coo_tensor(idx, vals, (0, 0), dtype=dtype, device=device).coalesce()

    if edge_index.numel() == 0:
        idx = torch.arange(num_nodes, device=device)
        indices = torch.stack([idx, idx], dim=0)
        values = torch.ones(num_nodes, dtype=dtype, device=device)
        return torch.sparse_coo_tensor(indices, values, (num_nodes, num_nodes), dtype=dtype, device=device).coalesce()

    edge_index = edge_index.to(device)
    # add self-loops
    self_loops = torch.arange(num_nodes, device=device)
    ei = torch.cat([edge_index, torch.stack([self_loops, self_loops], dim=0)], dim=1)

    # degree
    ones = torch.ones(ei.size(1), device=device, dtype=dtype)
    deg = torch.sparse_coo_tensor(ei, ones, (num_nodes, num_nodes), dtype=dtype, device=device)\
             .to_sparse_coo().sum(dim=1).to_dense()
    d_inv_sqrt = torch.pow(deg.clamp(min=1.0), -0.5)

    src, dst = ei[0], ei[1]
    norm_vals = d_inv_sqrt[src] * d_inv_sqrt[dst]

    A_norm = torch.sparse_coo_tensor(ei, norm_vals, (num_nodes, num_nodes), dtype=dtype, device=device).coalesce()
    return A_norm

def _mm_safely(adj_norm: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """
    Try sparse mm; if unsupported on the current backend (e.g., some MPS builds),
    fall back to dense mm. Always returns a dense tensor on the same device as x.
    """
    try:
        out = torch.sparse.mm(adj_norm, x)
        if out is None:
            # very rare path; just to be extra safe
            out = adj_norm.to_dense().mm(x)
        return out
    except Exception:
        return adj_norm.to_dense().mm(x)

class GraphConv(nn.Module):
    """H = Ĥ X W (GCN-style) with safe sparse/dense matmul."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)
    def forward(self, x, adj_norm):  # x: [N, F]
        xw = self.lin(x)              # [N, out]
        # ensure both are on the same device/dtype
        if adj_norm.device != xw.device:
            adj_norm = adj_norm.to(xw.device)
        if adj_norm.dtype != xw.dtype:
            adj_norm = adj_norm.to(dtype=xw.dtype)
        out = _mm_safely(adj_norm, xw)
        return out


In [7]:
# CLEAN PATCH for Cell 7 — Models (with attention, no truncation)

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class TemporalNodeEncoder(nn.Module):
    def __init__(self, in_dim=5, hidden=128, out=128, num_layers=2, bidirectional=True, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=num_layers,
                            batch_first=True, bidirectional=bidirectional,
                            dropout=dropout)
        self.proj = nn.Linear(hidden*(2 if bidirectional else 1), out)
        self.drop = nn.Dropout(dropout)
    def forward(self, x_seq):  # [N, SEQ, F]
        x_seq = x_seq.float()
        h, _ = self.lstm(x_seq)
        h_last = h[:, -1, :]
        return self.drop(self.proj(h_last))  # [N, out]

class GraphAttentionDense(nn.Module):
    def __init__(self, dim, heads=4, dropout=0.2):
        super().__init__()
        self.dim = dim
        self.heads = heads
        self.dk = dim // heads
        assert dim % heads == 0
        self.Wq = nn.Linear(dim, dim, bias=False)
        self.Wk = nn.Linear(dim, dim, bias=False)
        self.Wv = nn.Linear(dim, dim, bias=False)
        self.proj = nn.Linear(dim, dim)
        self.ln = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, H, mask, bias):
        # H: [N, dim], mask: [N,N], bias: [N,N]
        x = self.ln(H)
        N, d = x.shape
        q = self.Wq(x).view(N, self.heads, self.dk)
        k = self.Wk(x).view(N, self.heads, self.dk)
        v = self.Wv(x).view(N, self.heads, self.dk)

        logits = torch.einsum('nhd,mhd->hnm', q, k) / math.sqrt(self.dk)
        bias_h = bias.unsqueeze(0).expand(self.heads, N, N)
        logits = logits + 0.15 * bias_h
        neg_inf = torch.finfo(logits.dtype).min
        logits = torch.where(mask.unsqueeze(0) > 0.5,
                             logits,
                             torch.full_like(logits, neg_inf))
        attn = torch.softmax(logits, dim=-1)
        attn = self.drop(attn)
        out = torch.einsum('hnm,mhd->nhd', attn, v).reshape(N, d)
        out = self.drop(self.proj(out))
        return H + out

class GraphTransformerBlock(nn.Module):
    def __init__(self, dim, heads=4, mlp_ratio=2.0, dropout=0.2):
        super().__init__()
        self.attn = GraphAttentionDense(dim, heads=heads, dropout=dropout)
        self.ln = nn.LayerNorm(dim)
        hidden = int(dim*mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout)
        )
    def forward(self, H, mask, bias):
        H = self.attn(H, mask, bias)
        H = H + self.mlp(self.ln(H))
        return H

class LSTMOnlyClassifier(nn.Module):
    def __init__(self, in_dim=5, seq_len=30, enc_dim=128, num_classes=2, dropout=0.2):
        super().__init__()
        self.enc = TemporalNodeEncoder(in_dim, hidden=enc_dim, out=enc_dim, dropout=dropout)
        self.cls = nn.Sequential(
            nn.LayerNorm(enc_dim),
            nn.Dropout(dropout),
            nn.Linear(enc_dim, num_classes)
        )
    def forward(self, graph):
        dev = next(self.parameters()).device
        z = self.enc(graph["xseq"].to(dev))
        pooled = z.mean(dim=0, keepdim=True)
        return self.cls(pooled)

class GNNOnlyClassifier(nn.Module):
    def __init__(self, node_in=5, gdim=128, heads=4, num_classes=2, dropout=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(node_in), nn.Linear(node_in, gdim))
        self.block1 = GraphTransformerBlock(gdim, heads=heads, dropout=dropout)
        self.block2 = GraphTransformerBlock(gdim, heads=heads, dropout=dropout)
        self.cls = nn.Sequential(
            nn.LayerNorm(gdim),
            nn.Dropout(dropout),
            nn.Linear(gdim, num_classes)
        )
    def forward(self, graph):
        dev = next(self.parameters()).device
        xseq = graph["xseq"].to(dev)
        H0 = self.proj(xseq[:, -1, :].float())
        mask = graph["attn_mask"].to(dev)
        bias = graph["attn_bias"].to(dev)
        H = self.block1(H0, mask, bias)
        H = self.block2(H, mask, bias)
        pooled = H.mean(dim=0, keepdim=True)
        return self.cls(pooled)

class HybridTGNN(nn.Module):
    def __init__(self, in_dim=5, seq_len=30, enc_dim=128, gdim=128,
                 heads=4, num_classes=2, dropout=0.2):
        super().__init__()
        self.enc = TemporalNodeEncoder(in_dim, hidden=enc_dim, out=gdim, dropout=dropout)
        self.block1 = GraphTransformerBlock(gdim, heads=heads, dropout=dropout)
        self.block2 = GraphTransformerBlock(gdim, heads=heads, dropout=dropout)
        self.cls = nn.Sequential(
            nn.LayerNorm(gdim),
            nn.Dropout(dropout),
            nn.Linear(gdim, num_classes)
        )
    def forward(self, graph):
        dev = next(self.parameters()).device
        xseq = graph["xseq"].to(dev)
        mask = graph["attn_mask"].to(dev)
        bias = graph["attn_bias"].to(dev)
        H0 = self.enc(xseq)
        H = self.block1(H0, mask, bias)
        H = self.block2(H, mask, bias)
        pooled = H.mean(dim=0, keepdim=True)
        return self.cls(pooled)


In [8]:
# Cell 8 (REPLACE): accuracy-optimized threshold and selection

import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import (
    f1_score, precision_recall_curve, roc_auc_score, average_precision_score,
    roc_curve, accuracy_score
)

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=1.0, reduction="mean"):  # slightly smaller gamma favors accuracy
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction
    def forward(self, logits, target):
        ce = nn.functional.cross_entropy(logits, target, weight=self.weight, reduction="none")
        pt = torch.softmax(logits, dim=1).gather(1, target.view(-1,1)).squeeze(1).clamp(1e-6, 1-1e-6)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean() if self.reduction=="mean" else loss.sum()

def compute_class_weights(graphs, device):
    # For accuracy goals, don’t over-weight positives (keeps decision boundary closer to accuracy-optimal)
    pos = sum(int(g["y"].item()) for g in graphs)
    neg = len(graphs) - pos
    total = max(pos + neg, 1)
    w = torch.tensor([neg/total, pos/total], dtype=torch.float32, device=device)
    # Normalize to sum ~1 so the scale is stable
    w = w / (w.sum() + 1e-9)
    return w

def _best_accuracy_threshold(y_true, y_prob):
    # Use ROC curve thresholds to scan accuracy efficiently
    if len(np.unique(y_true)) < 2:
        return 0.5, 1.0  # degenerate
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    P = (y_true == 1).sum()
    N = (y_true == 0).sum()
    acc = (tpr * P + (1 - fpr) * N) / (P + N + 1e-9)
    idx = int(np.nanargmax(acc))
    return float(thr[idx]), float(acc[idx])

@torch.no_grad()
def evaluate(model, graphs, criterion=None, use_accuracy_thr=True, calibrator=None):
    model.eval()
    losses, ys, ps = [], [], []
    for g in graphs:
        logits = model(g)
        prob = torch.softmax(logits, dim=1)[0,1].item()
        y = int(g["y"].item())
        ps.append(prob); ys.append(y)
        if criterion is not None:
            y_tensor = torch.tensor([y], dtype=torch.long, device=next(model.parameters()).device)
            losses.append(criterion(logits, y_tensor).item())
    ys = np.array(ys); ps = np.array(ps)

    # Optional Platt calibration
    if calibrator is not None:
        ps = calibrator.transform(ps.reshape(-1,1)).ravel()

    # Choose threshold
    if use_accuracy_thr:
        thr, best_acc = _best_accuracy_threshold(ys, ps)
    else:
        # F1-optimal (fallback)
        if ys.sum()==0 or ys.sum()==len(ys):
            thr, best_acc = 0.5, accuracy_score(ys, (ps>=0.5).astype(int))
        else:
            prec, rec, thr_all = precision_recall_curve(ys, ps)
            f1s = 2*prec*rec/(prec+rec+1e-9)
            idx = int(np.nanargmax(f1s))
            thr = 0.5 if idx >= len(thr_all) else float(thr_all[idx])
            best_acc = accuracy_score(ys, (ps>=thr).astype(int))

    yhat = (ps >= thr).astype(int)

    acc = accuracy_score(ys, yhat)
    try:
        auroc = roc_auc_score(ys, ps) if len(np.unique(ys))>1 else 0.5
    except Exception:
        auroc = 0.5
    try:
        ap = average_precision_score(ys, ps) if ys.sum()>0 else 0.0
    except Exception:
        ap = 0.0
    f1 = f1_score(ys, yhat) if len(np.unique(ys))>1 else 0.0

    return {
        "loss": float(np.mean(losses)) if losses else None,
        "ACC": float(acc), "F1": float(f1),
        "AUROC": float(auroc), "AP": float(ap),
        "thr": float(thr), "ys": ys, "ps": ps
    }

def train_model_fixed(model, train_graphs, val_graphs,
                      epochs=60, virt_batch=16, lr=3e-4, weight_decay=1e-4, gamma=1.0):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(dev)
    weights = compute_class_weights(train_graphs, dev)
    criterion = FocalLoss(weight=weights, gamma=gamma)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best = {"ACC": -1.0, "state": None, "epoch": -1, "val_summary": None}
    for ep in range(1, epochs+1):
        model.train()
        total_loss = 0.0
        opt.zero_grad(set_to_none=True)
        for i, g in enumerate(train_graphs):
            logits = model(g)
            y = g["y"].to(dev)
            loss = criterion(logits, y)
            loss.backward()
            if (i+1) % virt_batch == 0:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                opt.zero_grad(set_to_none=True)
            total_loss += loss.item()
        if len(train_graphs) % virt_batch != 0:
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            opt.zero_grad(set_to_none=True)

        tr = evaluate(model, train_graphs, criterion, use_accuracy_thr=True)
        va = evaluate(model, val_graphs,   criterion, use_accuracy_thr=True)
        print(f"[{ep:03d}] train_loss={tr['loss']:.4f}  val_loss={va['loss']:.4f}  "
              f"val_ACC={va['ACC']:.3f}  val_F1={va['F1']:.3f}  val_AP={va['AP']:.3f}  val_AUROC={va['AUROC']:.3f}")

        if va["ACC"] > best["ACC"]:
            best.update(ACC=va["ACC"], state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                        epoch=ep, val_summary=va)

    model.load_state_dict(best["state"])
    return model, best["val_summary"]


In [9]:
# Calibration (Platt scaling) fitted on validation, applied to test
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

class PlattCalibrator:
    def __init__(self, max_iter=200):
        self.lr = LogisticRegression(max_iter=max_iter)
        self.fitted = False
    def fit(self, y_prob, y_true):
        y_prob = np.asarray(y_prob).reshape(-1,1)
        y_true = np.asarray(y_true).ravel()
        self.lr.fit(y_prob, y_true)
        self.fitted = True
    def transform(self, y_prob_2d):
        if not self.fitted:
            return y_prob_2d.ravel()
        return self.lr.predict_proba(y_prob_2d)[:,1].reshape(-1,1)

def evaluate_with_calibration(model, train_graphs, val_graphs, test_graphs, criterion=None):
    # 1) Raw validation probabilities
    val_raw = evaluate(model, val_graphs, criterion, use_accuracy_thr=True, calibrator=None)
    # 2) Fit calibrator on validation set
    cal = PlattCalibrator()
    cal.fit(val_raw["ps"], val_raw["ys"])
    # 3) Re-evaluate on validation to get calibrated, accuracy-optimal threshold
    val_cal = evaluate(model, val_graphs, criterion, use_accuracy_thr=True, calibrator=cal)
    thr = val_cal["thr"]
    # 4) Evaluate on test with same calibrator & thr
    test_cal = evaluate(model, test_graphs, criterion, use_accuracy_thr=True, calibrator=cal)
    yhat = (test_cal["ps"] >= thr).astype(int)
    acc  = accuracy_score(test_cal["ys"], yhat)
    f1   = f1_score(test_cal["ys"], yhat) if len(np.unique(test_cal["ys"]))>1 else 0.0
    auroc= roc_auc_score(test_cal["ys"], test_cal["ps"]) if len(np.unique(test_cal["ys"]))>1 else 0.5
    ap   = average_precision_score(test_cal["ys"], test_cal["ps"]) if test_cal["ys"].sum()>0 else 0.0
    return {"ACC": acc, "F1": f1, "AUROC": auroc, "AP": ap, "thr": thr}


In [10]:
# Train all models with identical budget 

cfg = {"in_dim": len(FEATS), "seq_len": SEQ, "enc_dim": 128,
       "gdim": 128, "num_classes": 2}

models = {
    "LSTM_only": LSTMOnlyClassifier(
        **{k: cfg[k] for k in ["in_dim","seq_len","enc_dim","num_classes"]},
        dropout=0.20),
    "GNN_only":  GNNOnlyClassifier(
        node_in=cfg["in_dim"], gdim=cfg["gdim"],
        heads=4, num_classes=2, dropout=0.20),
    "Hybrid":    HybridTGNN(**cfg, heads=4, dropout=0.15),  # tiny reg tweak helps accuracy
}

trained = {}
for name, mdl in models.items():
    print("\n==== Training", name, "====")
    m, best = train_model_fixed(
        mdl, train_graphs, val_graphs,
        epochs=30, virt_batch=16, lr=3e-4, weight_decay=1e-4, gamma=1.0
    )
    trained[name] = (m, best)

print("\nValidation (accuracy-picked) summary:")
for name, (_, best) in trained.items():
    print(f"{name:10s} | ACC={best['ACC']:.3f}  F1={best['F1']:.3f}  AP={best['AP']:.3f}  AUROC={best['AUROC']:.3f}  thr={best['thr']:.3f}")



==== Training LSTM_only ====
[001] train_loss=0.1713  val_loss=0.1777  val_ACC=0.629  val_F1=0.298  val_AP=0.474  val_AUROC=0.483
[002] train_loss=0.1789  val_loss=0.1789  val_ACC=0.640  val_F1=0.360  val_AP=0.470  val_AUROC=0.434
[003] train_loss=0.1696  val_loss=0.1728  val_ACC=0.640  val_F1=0.360  val_AP=0.457  val_AUROC=0.430
[004] train_loss=0.1683  val_loss=0.1743  val_ACC=0.629  val_F1=0.327  val_AP=0.454  val_AUROC=0.410
[005] train_loss=0.1684  val_loss=0.1749  val_ACC=0.629  val_F1=0.327  val_AP=0.475  val_AUROC=0.423
[006] train_loss=0.1683  val_loss=0.1740  val_ACC=0.629  val_F1=0.353  val_AP=0.484  val_AUROC=0.446
[007] train_loss=0.1690  val_loss=0.1760  val_ACC=0.640  val_F1=0.360  val_AP=0.490  val_AUROC=0.459
[008] train_loss=0.1680  val_loss=0.1749  val_ACC=0.629  val_F1=0.353  val_AP=0.470  val_AUROC=0.439
[009] train_loss=0.1683  val_loss=0.1747  val_ACC=0.618  val_F1=0.393  val_AP=0.485  val_AUROC=0.445
[010] train_loss=0.1685  val_loss=0.1762  val_ACC=0.629  val_

In [11]:
print("\nCalibrated test accuracy:")
acc_scores = {}
for name, (m, _) in trained.items():
    cal_test = evaluate_with_calibration(m, train_graphs, val_graphs, test_graphs, criterion=None)
    acc_scores[name] = cal_test
    print(f"{name:10s} | ACC={cal_test['ACC']:.3f}  F1={cal_test['F1']:.3f}  AUROC={cal_test['AUROC']:.3f}  AP={cal_test['AP']:.3f}  (thr={cal_test['thr']:.3f})")

# Winner by Accuracy (tie-breaker: AUROC, then F1)
def acc_sort_key(item):
    s = item[1]
    return (s["ACC"], s["AUROC"], s["F1"])

winner, wscore = sorted(acc_scores.items(), key=acc_sort_key, reverse=True)[0]
print("\n WINNER by Accuracy:", winner,
      "| ACC=%.3f  AUROC=%.3f  F1=%.3f  AP=%.3f" %
      (wscore["ACC"], wscore["AUROC"], wscore["F1"], wscore["AP"]))



Calibrated test accuracy:
LSTM_only  | ACC=0.466  F1=0.299  AUROC=0.448  AP=0.461  (thr=0.427)
GNN_only   | ACC=0.477  F1=0.258  AUROC=0.399  AP=0.428  (thr=0.431)
Hybrid     | ACC=0.500  F1=0.290  AUROC=0.481  AP=0.457  (thr=0.427)

 WINNER by Accuracy: Hybrid | ACC=0.500  AUROC=0.481  F1=0.290  AP=0.457


In [12]:
# Cell 10: Test evaluation using the threshold selected on validation per model

def test_with_val_thr(model, graphs, val_summary):
    model.eval()
    _, ys, ps = [], [], []
    out = evaluate(model, graphs)  # compute fresh metrics too
    thr = val_summary["thr"]
    ys = out["ys"]
    ps = out["ps"]
    yhat = (ps >= thr).astype(int)
    f1 = f1_score(ys, yhat) if len(np.unique(ys)) > 1 else 0.0
    try:
        auroc = roc_auc_score(ys, ps) if len(np.unique(ys))>1 else 0.5
    except Exception:
        auroc = 0.5
    try:
        ap = average_precision_score(ys, ps) if ys.sum()>0 else 0.0
    except Exception:
        ap = 0.0
    return {"F1": f1, "AP": ap, "AUROC": auroc, "thr": float(thr)}

print("Test results:")
for name, (m, va_best) in trained.items():
    te = test_with_val_thr(m, test_graphs, va_best)
    print(f"{name:10s} | F1={te['F1']:.3f}  AP={te['AP']:.3f}  AUROC={te['AUROC']:.3f}  (thr from val={te['thr']:.3f})")


Test results:
LSTM_only  | F1=0.333  AP=0.516  AUROC=0.552  (thr from val=0.409)
GNN_only   | F1=0.258  AP=0.428  AUROC=0.399  (thr from val=0.435)
Hybrid     | F1=0.207  AP=0.478  AUROC=0.519  (thr from val=0.384)
